# Substrate-Induced Band Gap of Graphene on h-BN

> **Gianluca Giovannetti, Petr A. Khomyakov, Geert Brocks, Paul J. Kelly, and Jeroen van den Brink**
> **Substrate-induced band gap in graphene on hexagonal boron nitride: Ab initio density functional calculations**
> Physical Review B 76, 073103 (2007)
> [DOI: 10.1103/PhysRevB.76.073103](https://doi.org/10.1103/PhysRevB.76.073103)

Calculate the total energy and the band structure of graphene on four layers of h-BN for the three
stackings and the list of graphene–h-BN distances created in the
[structure notebook](interface_2d_2d_boron_nitride_graphene.ipynb), using Quantum ESPRESSO and the
band structure + density of states workflow from Standata. The results reproduce Giovannetti et al.
(2007): Fig. 2 (total energy vs distance and the equilibrium distances), Fig. 3 (bands and density of
states of stacking (c) at its equilibrium distance, gap at K) and Fig. 4 (gap at K vs distance).

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](interface_2d_2d_boron_nitride_graphene.ipynb), which saves them to the `uploads` folder under the names built from `MATERIAL_NAME` in cell 1.2 below.
1. Set the stackings, distances and calculation parameters in cells 1.2 and 1.3, or keep the default values: the active stackings × distances run with them. Uncomment the rest of `STACKINGS` and `DISTANCES`, here and in the structure notebook, for the paper's full set.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results; section 8 prints the comparison with the paper.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the stackings, distances, workflow, model, compute resources and jobs.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the materials by name from the `uploads` folder, print their provenance and save them to the platform.
1. Configure the workflow: select the application and the model, load the band structure + DOS workflow from Standata, and set the computational parameters, the tetrahedron occupations and the dipole correction for each material.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Run one job per material, one after another, re-using a job that already ran under the same workflow name.
1. Retrieve results: total energies, the direct gap at K, the equilibrium distance of each stacking, the total energy and the gap vs distance, the bands and density of states of (c) at its equilibrium distance, the h-BN gap and the effective mass at K.
1. Compare with Giovannetti et al. (2007).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters and configurations for the workflow and jobs

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

STACKINGS = [
    "c",
    # "a",
    # "b",
]
DISTANCES = [
    3.1, 3.2, 3.3,
    # 2.5, 2.6, 2.7, 2.8, 2.9, 3.0, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9,
]
MATERIAL_NAME = "Gr/hBN ({stacking}) d{distance:.2f}"  # as saved by the structure notebook

WORKFLOW_SEARCH_TERM = "band_structure_dos.json"
MY_WORKFLOW_NAME = "Band Structure + DOS"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = "001"  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.3. Set the DFT parameters

In [ ]:
MODEL_SUBTYPE = "lda"
FUNCTIONAL = "pz"  # Giovannetti et al. 2007 use LDA: GGA gives essentially no interlayer binding
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft, the only LDA family the platform publishes for B, C and N
ECUTWFC = 40  # Ry, GBRV's tested cutoff; the paper's 600 eV is a VASP number
ECUTRHO = 200  # Ry, GBRV's tested charge-density cutoff

KGRID = [36, 36, 1]  # Giovannetti et al. 2007; a multiple of 3 keeps K on the mesh
OCCUPATIONS_SETTINGS = {"occupations": "tetrahedra"}  # the paper's tetrahedron method
DIPOLE_SETTINGS = {"control": {"tefield": True, "dipfield": True}, "system": {"edir": 3, "eopreg": 0.05}}
KPATH_STEPS = 100
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KGRID[0]} p{KPATH_STEPS} tetra dip"

SCF_UNIT = "pw_scf"
NSCF_UNIT = "pw_nscf"
BANDS_UNIT = "pw_bands"
NUMBER_OF_OCCUPIED_BANDS = 20  # 40 valence electrons: C 4×2, B 3×4, N 5×4

KPATH = [
    {"point": "Γ", "steps": KPATH_STEPS},
    {"point": "K", "steps": KPATH_STEPS},
    {"point": "M", "steps": KPATH_STEPS},
    {"point": "Γ", "steps": 1},
]

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load from the uploads folder and print provenance

The structures, their geometry and their lattice type all come from the structure notebook; this one
only loads them by name, one per stacking and distance.

In [ ]:
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from mat3ra.notebooks_utils.core.entity.material.api import load_material

material_names = {
    stacking: {distance: MATERIAL_NAME.format(stacking=stacking, distance=distance) for distance in sorted(DISTANCES)}
    for stacking in STACKINGS
}
materials = {}
for stacking, names in material_names.items():
    for name in names.values():
        material = load_material(client, FOLDER, name, ACCOUNT_ID)
        materials[name] = material
        interlayer_distance = get_average_interlayer_distance(
            material, InterfacePartsEnum.SUBSTRATE.value, InterfacePartsEnum.FILM.value)
        print(f"{name}: {material.basis.number_of_atoms} atoms, a = {material.lattice.a:.4f} Å, "
              f"gamma = {material.lattice.gamma:.3f}°, graphene–h-BN distance = {interlayer_distance:.3f} Å, "
              f"stacking ({stacking})")

### 3.2. Preview the materials

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

visualize_materials([{"material": material, "title": name} for name, material in materials.items()],
                    viewer="wave")

### 3.3. Save the materials to the platform

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = {}
for name, material in materials.items():
    material.basis.set_labels_from_list([])
    saved_materials[name] = Material.create(get_or_create_material(client, material, ACCOUNT_ID))

## 4. Configure the workflow
### 4.1. Select application and model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")

### 4.2. Configure the workflow

One workflow per material, all with the settings of 1.3. Each workflow is named after its material
and `MODEL_TAG`; section 6 finds a job that already ran by that name.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider, PointsGridDataProvider, \
    PointsPathDataProvider
from mat3ra.notebooks_utils.workflow import patch_workflow_qe_input

from copy import deepcopy

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)

cutoffs_context = PlanewaveCutoffsContextProvider(
    wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True).get_context_item_data()
grid_context = PointsGridDataProvider(
    material=next(iter(materials.values())), dimensions=KGRID, isEdited=True).get_context_item_data()
path_context = PointsPathDataProvider(path=KPATH, isEdited=True).get_context_item_data()


def get_vacuum_center(material):
    return (max(coordinate[2] for coordinate in material.basis.coordinates.values) + 1) / 2


workflows = {}
for name, material in materials.items():
    workflow = Workflow.create(deepcopy(workflow_config))
    workflow.name = f"{MY_WORKFLOW_NAME} {name} {MODEL_TAG}"
    subworkflow = workflow.subworkflows[0]
    subworkflow.model = model

    for unit_name, contexts in [(SCF_UNIT, [grid_context, cutoffs_context]),
                                (NSCF_UNIT, [grid_context, cutoffs_context]),
                                (BANDS_UNIT, [path_context, cutoffs_context])]:
        unit = subworkflow.get_unit_by_name(name=unit_name)
        for context in contexts:
            unit.add_context(context)
        subworkflow.set_unit(unit)
    patch_workflow_qe_input(workflow, DIPOLE_SETTINGS, [SCF_UNIT, NSCF_UNIT, BANDS_UNIT])
    patch_workflow_qe_input(workflow, {"system": OCCUPATIONS_SETTINGS}, [SCF_UNIT, NSCF_UNIT])
    patch_workflow_qe_input(workflow, {"system": {"emaxpos": get_vacuum_center(material)}},
                            [SCF_UNIT, NSCF_UNIT, BANDS_UNIT])
    workflows[name] = workflow
    print(workflow.name)

### 4.3. Preview the workflow

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

visualize_workflow(next(iter(workflows.values())))

### 4.4. Save the workflows to the collection

In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

for name, workflow in workflows.items():
    saved = Workflow.create(get_or_create_workflow(client, workflow, ACCOUNT_ID))
    print(f"{name}: workflow {saved.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Run the jobs one at a time

A job is found by its material and its workflow name, which carries the model tag, and re-used if it
exists; otherwise it is created and submitted. The jobs run one after another.

In [ ]:
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

job_ids = {}
for name, workflow in workflows.items():
    job = find_job_for_material(
        client, saved_materials[name].id, workflow.name, ACCOUNT_ID,
        statuses=("submitted", "queued", "active", "finished"),
    )
    if job is None:
        job = create_job(
            api_client=client, materials=[saved_materials[name]], workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
        )
        submit_jobs(client.jobs, [job["_id"]])
        print(f"✅ {name}: submitted job {job['_id']}")
    else:
        print(f"♻️  {name}: reusing job {job['_id']}")
    await wait_for_jobs_to_finish_async(client.jobs, [job["_id"]], poll_interval=POLL_INTERVAL)
    job_ids[name] = job["_id"]

## 7. Retrieve the results
### 7.1. Total energies

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job

total_energies = {}
for name, job_id in job_ids.items():
    total_energies[name] = get_properties_for_job(client, job_id, "total_energy")[0]["value"]
    print(f"{name}: {total_energies[name]:.6f} eV")

### 7.2. Direct gap at K

The difference between the lowest unoccupied and the highest occupied band at K, the end of the first
leg of `KPATH`.

In [ ]:
import numpy as np

K_VERTEX_INDEX = KPATH_STEPS

band_structures = {}
gaps = {}
for name, job_id in job_ids.items():
    band_structures[name] = get_properties_for_job(client, job_id, property_name="band_structure")[0]
    bands_at_k = np.array(band_structures[name]["yDataSeries"])[:, K_VERTEX_INDEX]
    gaps[name] = 1000 * (bands_at_k[NUMBER_OF_OCCUPIED_BANDS] - bands_at_k[NUMBER_OF_OCCUPIED_BANDS - 1])
    print(f"{name}: {gaps[name]:.1f} meV")

### 7.3. Equilibrium distance of each stacking

The vertex of the parabola through the three distances with the lowest total energy; the gap at K is
printed for the distance in `DISTANCES` nearest to it.

In [ ]:
equilibrium_distances = {}
equilibrium_names = {}
for stacking, names in material_names.items():
    lowest = sorted(names, key=lambda distance: total_energies[names[distance]])[:3]
    quadratic, linear, _ = np.polyfit(lowest, [total_energies[names[distance]] for distance in lowest], 2)
    equilibrium_distances[stacking] = -linear / (2 * quadratic)
    nearest = min(names, key=lambda distance: abs(distance - equilibrium_distances[stacking]))
    equilibrium_names[stacking] = names[nearest]
    print(f"({stacking}): equilibrium distance {equilibrium_distances[stacking]:.3f} Å, "
          f"gap at K at {nearest:.2f} Å: {gaps[names[nearest]]:.1f} meV")

### 7.4. Total energy vs distance (Fig. 2 of the paper)

In [ ]:
from mat3ra.notebooks_utils.plot import plot_series

for stacking, names in material_names.items():
    reference_energy = total_energies[names[max(names)]]
    series = [
        {"distance": distance, "energy": total_energies[name] - reference_energy} for distance, name in names.items()
    ]
    plot_series(series=series, x_key="distance", y_key="energy", xlabel="Graphene–h-BN distance (Å)",
                ylabel=f"E − E({max(names):.2f} Å) (eV per cell)",
                title=f"Total energy vs distance, stacking ({stacking})")

### 7.5. Gap at K vs distance (Fig. 4 of the paper)

In [ ]:
for stacking, names in material_names.items():
    series = [{"distance": distance, "gap": gaps[name]} for distance, name in names.items()]
    plot_series(series=series, x_key="distance", y_key="gap", xlabel="Graphene–h-BN distance (Å)",
                ylabel="Direct gap at K (meV)", title=f"Gap at K vs distance, stacking ({stacking})")

### 7.6. Bands and density of states of (c) at its equilibrium distance (Fig. 3 of the paper)

The job at the distance nearest to the equilibrium of (c): band structure, density of states, and the
two bands around the gap near K.

In [ ]:
from matplotlib import pyplot as plt
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

name = equilibrium_names["c"]
visualize_properties(band_structures[name], title=f"Band Structure: {name}",
                     extra_config={"material": materials[name].to_dict()})
visualize_properties(get_properties_for_job(client, job_ids[name], property_name="density_of_states"),
                     title=f"Density of States: {name}", extra_config={"material": materials[name].to_dict()})

bands = np.array(band_structures[name]["yDataSeries"])
path_indices = np.arange(K_VERTEX_INDEX - 12, K_VERTEX_INDEX + 13)
center = bands[NUMBER_OF_OCCUPIED_BANDS - 1 : NUMBER_OF_OCCUPIED_BANDS + 1, K_VERTEX_INDEX].mean()
figure, axes = plt.subplots()
for band in bands[NUMBER_OF_OCCUPIED_BANDS - 1 : NUMBER_OF_OCCUPIED_BANDS + 1]:
    axes.plot(path_indices, band[path_indices])
axes.set_ylim(center - 0.5, center + 0.5)
axes.set_xlabel(f"Path index (K at {K_VERTEX_INDEX})")
axes.set_ylabel("Energy (eV)")
axes.set_title("Bands around K")
display_matplotlib_figure(figure)

### 7.7. h-BN gap at K

In [ ]:
# the h-BN bands adjacent to the graphene pair
h_bn_gap = bands[NUMBER_OF_OCCUPIED_BANDS + 1, K_VERTEX_INDEX] - bands[NUMBER_OF_OCCUPIED_BANDS - 2, K_VERTEX_INDEX]
print(f"{name}: h-BN gap at K {h_bn_gap:.2f} eV")

### 7.8. Effective mass at K

For a gapped Dirac cone, E(k) = ±√((Δ/2)² + (ħv k)²), the band-edge mass is m* = ħ²Δ / (2(ħv)²), with Δ
the gap at K from 7.2 and ħv the slope of the lowest unoccupied band against |k − K| on the Γ–K leg.

In [ ]:
HBAR_SQUARED_OVER_ELECTRON_MASS = 7.6200  # eV·Å²
K_STEP = 4 * np.pi / (3 * materials[name].lattice.a) / KPATH_STEPS  # Å⁻¹ per path step along Γ–K

k_offsets = K_STEP * np.arange(8, 2, -1)
hbar_velocity, _ = np.polyfit(k_offsets, bands[NUMBER_OF_OCCUPIED_BANDS, K_VERTEX_INDEX - 8 : K_VERTEX_INDEX - 2], 1)
effective_mass = HBAR_SQUARED_OVER_ELECTRON_MASS * gaps[name] / 1000 / (2 * hbar_velocity**2)
print(f"{name}: ħv = {hbar_velocity:.2f} eV·Å, effective mass at K {effective_mass:.2e} m_e")

## 8. Compare with Giovannetti et al. (2007)

The paper's equilibrium distances (Fig. 2) and gaps at equilibrium (Fig. 4, p. 3) for each active
stacking, then the h-BN gap and the effective mass at K for (c), printed beside the values above.

Settings that differ from the paper's: GBRV ultrasoft pseudopotentials at 40/200 Ry against VASP at 600 eV.

In [ ]:
GIOVANNETTI_EQUILIBRIUM_DISTANCE = {"a": 3.50, "b": 3.40, "c": 3.22}  # Å, Fig. 2
GIOVANNETTI_GAP_AT_EQUILIBRIUM = {"a": 56, "b": 46, "c": 53}  # meV, Fig. 4 / p.3
GIOVANNETTI_H_BN_GAP_AT_K = 4.7  # eV, p.3
GIOVANNETTI_EFFECTIVE_MASS = 4.7e-3  # m_e, (c), p.4

print(f"{'stacking':>8} {'d_eq (Å)':>9} {'paper':>6} {'deviation':>9} {'gap at d_eq (meV)':>18} {'paper':>6} "
      f"{'deviation':>9}")
for stacking in STACKINGS:
    distance = equilibrium_distances[stacking]
    paper_distance = GIOVANNETTI_EQUILIBRIUM_DISTANCE[stacking]
    gap = gaps[equilibrium_names[stacking]]
    paper_gap = GIOVANNETTI_GAP_AT_EQUILIBRIUM[stacking]
    print(f"{stacking:>8} {distance:>9.3f} {paper_distance:>6.2f} "
          f"{(distance - paper_distance) / paper_distance:>+9.1%} "
          f"{gap:>18.1f} {paper_gap:>6} {(gap - paper_gap) / paper_gap:>+9.1%}")

print(f"h-BN gap at K, (c): {h_bn_gap:.2f} eV, paper {GIOVANNETTI_H_BN_GAP_AT_K} eV")
print(f"Effective mass at K, (c): {effective_mass:.2e} m_e, paper {GIOVANNETTI_EFFECTIVE_MASS:.1e} m_e")

energy_order = [stacking for stacking in ["c", "b", "a"] if stacking in STACKINGS]
energy_order_holds = all(
    total_energies[material_names[lower][distance]] < total_energies[material_names[higher][distance]]
    for lower, higher in zip(energy_order, energy_order[1:])
    for distance in DISTANCES
)
print(f"E(c) < E(b) < E(a) at every distance: {energy_order_holds}")

## References

[1] G. Giovannetti, P. A. Khomyakov, G. Brocks, P. J. Kelly and J. van den Brink, "Substrate-induced band gap in graphene on hexagonal boron nitride: Ab initio density functional calculations", Phys. Rev. B 76, 073103 (2007). https://doi.org/10.1103/PhysRevB.76.073103